# SatQuery AI — Phase 5 training campaign

**This notebook holds no state.** Every cell can be re-run from scratch after the
kernel dies, and nothing is lost but the seconds since the last checkpoint. That is
the entire design: a JupyterHub session will be killed — on idle timeout, on the
wall clock, when the tab closes, when the cluster reclaims the node — and a campaign
that keeps its progress in the kernel loses it several times a week.

What holds the state instead:

| | where | survives a kill? |
|---|---|---|
| which runs are done | `runs/campaign_state.json` | yes |
| how far each run got | `checkpoints/v2/<run>/ckpt_step_*.pt` | yes |
| what happened at 3 a.m. | `runs/logs/<run>.log` | yes |

## The loop

1. run cells 1–4 once at the start of a session (they are cheap and idempotent);
2. run **cell 5**, the training cell, and leave it;
3. when the session dies, open the notebook again and **run the same cells again**.

Cell 5 reclaims any run the dead session left marked `running`, resumes it, and
carries on. You do not have to remember what was running.

## 1. Locate the repository

Set `REPO` if the notebook is not inside a checkout. Everything else is derived,
so this is the only path in the notebook.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = Path(os.environ.get("SATQUERY_REPO", Path.cwd().parent)).resolve()
assert (REPO / "training" / "cluster").is_dir(), f"not a SatQuery checkout: {REPO}"
os.chdir(REPO)
sys.path.insert(0, str(REPO))

# Datasets usually live on scratch, not in the repo, because home directories on
# a shared cluster have quotas an order of magnitude below 63 GB.
DATA_ROOT = Path(os.environ.get("SATQUERY_DATA_ROOT", REPO / "data"))
print(f"repo  {REPO}\ndata  {DATA_ROOT}")

## 2. What card did we get?

A shared cluster hands out whatever is free, so this is checked every session
rather than assumed. The probe decides precision (bf16 needs compute capability
8.0), attention implementation, and the batch shape — with the **effective**
batch held constant across cards, so a bigger GPU makes a run faster rather
than different.

In [ ]:
from training.cluster.env_probe import probe, recipe_for, BASE_RECIPES

gpu = probe(DATA_ROOT)
print(f"{gpu.name} x{gpu.device_count}   {gpu.vram_gb:.1f} GB   "
      f"cap {gpu.capability}   {gpu.dtype_name}   {gpu.attn_implementation}")
print(f"free disk on the data mount: {gpu.free_disk_gb:.0f} GB")
for note in gpu.notes:
    print(f"  note: {note}")

if not gpu.available:
    print("\nNO GPU VISIBLE. Check the kernel is on a GPU node before running cell 5.")

for family in BASE_RECIPES:
    r = recipe_for(family, gpu)
    print(f"  {family:<12} micro={r['micro_batch']:<4} accum={r['grad_accum']:<3} "
          f"effective={r.get('effective_batch', '-')}")

## 3. Is the data actually here, and intact?

Digests, not sizes. A truncated HDF5 shard opens fine, reads its first shard
fine, and kills the run four hours in — or worse, does not kill it and trains a
short epoch. This project has already lost files to a restore that returned NUL
bytes and a verification that hashed without opening (`docs/model-cards.md`).

The full check on 63 GB takes a few minutes. `plan` reports which runs the data
present so far already unblocks, so a partial transfer does not have to block
the whole campaign.

In [ ]:
import json
from training.cluster.stage_data import verify, runs_unblocked

manifest_path = REPO / "configs" / "data_manifest.json"
if not manifest_path.is_file():
    print("No manifest. Build one on the machine that prepared the data:")
    print("  python training/cluster/stage_data.py build --root data")
else:
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    verdicts = verify(DATA_ROOT, manifest)
    for v in verdicts.values():
        print(v.summary())
    ready, blocked = runs_unblocked(verdicts)
    print(f"\nunblocked: {', '.join(ready) or '(none)'}")
    if blocked:
        print(f"blocked:   {', '.join(blocked)}")

## 4. Where the campaign stands

In [ ]:
from training.cluster.campaign import load_campaign

campaign = load_campaign("configs/campaign.yaml", "runs")
campaign.reclaim_stale()   # re-queue anything the last dead session left running
print(campaign.report())

## 5. Train

**This is the cell you re-run after every session death.** It picks the next run
whose dependencies are satisfied, resumes it if it has been started before, and
streams its output here and to `runs/logs/<run>.log`.

`SESSION_MINUTES` is what your cluster gives you. It is **advisory**: the queue
prefers a run that fits, and starts a longer one anyway when none does, because
every trainer checkpoints every few hundred steps and partial progress is kept.
Pass `strict=True` only if you must hand the node back on time.

In [ ]:
SESSION_MINUTES = 8 * 60   # set to your cluster's wall-clock limit

failures = campaign.run_all(budget_minutes=SESSION_MINUTES)
print(campaign.report())
if failures:
    print(f"\n{failures} run(s) FAILED - read the log before re-running. A failed run "
          "is not retried automatically, because burning GPU hours reproducing the "
          "same traceback is worse than stopping.")

### Running one job on its own

For a first smoke run, or to retry one thing after fixing it.

In [ ]:
# campaign.launch("change_mask")                 # cheapest real run; good first test
# campaign.launch("track_a", dry_run=True)       # print the command, run nothing
# campaign.state["grounding"].status = "pending"; campaign.save()   # retry a failure

## 6. What did it produce?

Each finished run writes `metrics.json` and `run_metadata.json` into its
checkpoint directory. These are the numbers that go into the model cards — and
under `docs/code-freeze.md` they are published as a **new dated section**, never
as an edit to the v1 numbers. The v1 result is what the v2 result is compared
against; overwriting it destroys the comparison.

In [ ]:
for run_id, run in campaign.runs.items():
    metrics = REPO / run.ckpt_dir / "metrics.json"
    if not metrics.is_file():
        print(f"{run_id:<22} (no metrics yet)")
        continue
    data = json.loads(metrics.read_text(encoding="utf-8"))
    headline = {k: v for k, v in data.items() if isinstance(v, (int, float))}
    print(f"{run_id:<22} {headline}")